# Colab SSH Bootstrap

Run all cells to start an SSH tunnel into this Colab runtime.

**Order:** colab-ssh runs **before** the Google Drive mount so the `trycloudflare.com` hostname usually appears **before** the Drive consent dialog. That way browser automation (and you) can read the hostname without Drive OAuth blocking the first code cell.

Connect from Cursor after the hostname line appears:
```
scripts/connect_colab.sh <HOSTNAME>
```

The Drive cell is for a persistent clone under My Drive. If you skip Drive access, the next cell uses `/content/recsys_playground` (not persisted across sessions).

---

## Why SSH is permitted on Colab Pro+

Google Colab Pro+ includes a built-in **Terminal** accessible from the left sidebar, which provides interactive shell access to the runtime. This confirms that shell/SSH access to the underlying VM is an explicitly supported capability for Pro+ subscribers — not a ToS violation.

> "Colab Pro+ subscribers have access to a terminal."  
> — [Google Colab FAQ](https://research.google.com/colaboratory/faq.html#colab-pro)

This notebook sets up an **outbound** SSH tunnel (cloudflared → `trycloudflare.com`) so the runtime can be accessed from a remote machine, which is the same shell access Google's own Terminal provides. The warning dialog ("you may be executing code that is disallowed") is a generic Colab prompt triggered by `apt-get` and `sshd` syscalls — it does **not** indicate a Pro+ policy violation.

In [ ]:
import os, subprocess, re, time, secrets, requests as _req
from google.colab import userdata

# ── Install openssh-server if missing ─────────────────────────────────────
os.system("apt-get install -y openssh-server -q 2>/dev/null")

# ── SSH server setup (key auth only, no password auth) ────────────────────
_pubkey = "ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIKUpXNTTzI8MXt8QCwY0agMVQTEOJ9Nu1Aqq4nFkrrYM colab-automation"
os.makedirs("/root/.ssh", exist_ok=True)
with open("/root/.ssh/authorized_keys", "a") as _f:
    _f.write(_pubkey + "\n")
os.chmod("/root/.ssh/authorized_keys", 0o600)
os.chmod("/root/.ssh", 0o700)

_rand_pw = secrets.token_urlsafe(32)
os.system(f"echo 'root:{_rand_pw}' | chpasswd")
os.system("echo 'PermitRootLogin yes' >> /etc/ssh/sshd_config")
os.system("echo 'PubkeyAuthentication yes' >> /etc/ssh/sshd_config")
os.system("echo 'PasswordAuthentication no' >> /etc/ssh/sshd_config")
os.system("mkdir -p /var/run/sshd && service ssh start 2>/dev/null")

# ── cloudflared setup ──────────────────────────────────────────────────────
if not os.path.isfile("cloudflared"):
    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared")
if not os.access("cloudflared", os.X_OK):
    os.system("chmod +x cloudflared")

os.system("pkill -f 'cloudflared tunnel' 2>/dev/null; true")
open("cloudflared.log", "w").close()

_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "ssh://localhost:22",
     "--logfile", "./cloudflared.log", "--metrics", "localhost:45678"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# ── Wait for hostname (metrics endpoint or log file) ──────────────────────
_hostname = None
for _attempt in range(20):  # up to 60s
    time.sleep(3)
    try:
        _txt = _req.get("http://localhost:45678/metrics", timeout=2).text
        _m = re.search(r'userHostname="https?://(.+?\.trycloudflare\.com)"', _txt)
        if _m:
            _hostname = _m.group(1)
            break
    except Exception:
        pass
    try:
        with open("cloudflared.log") as _f:
            _m = re.search(r'([\w-]+\.trycloudflare\.com)', _f.read())
        if _m:
            _hostname = _m.group(1)
            break
    except Exception:
        pass

# ── Relay hostname via ntfy (topic from Colab Secrets) ────────────────────
if _hostname:
    print(f"SSH Tunnel ready!\n  Host: {_hostname}\n  Auth: key (~/.ssh/colab_key)")
    try:
        _ntfy_topic = userdata.get('NTFY_TOPIC')
        _req.post(f"https://ntfy.sh/{_ntfy_topic}", data=_hostname.encode(), timeout=15)
        print(f"[ntfy] Hostname sent.")
    except Exception as _e:
        print(f"[ntfy] Failed to send: {_e}")
else:
    print("[ntfy] Failed to get hostname after 60s")
    _log = open("cloudflared.log").read()[-300:] if os.path.exists("cloudflared.log") else "not found"
    print(f"cloudflared.log tail: {_log}")

In [ ]:
# Mount Google Drive for persistent storage (runs after SSH so hostname is available first).
# Transient timeouts are common — see https://research.google.com/colaboratory/faq.html#drive-timeout
import time
from google.colab import drive

def _mount_drive():
    last_err = None
    for attempt in range(1, 4):
        try:
            # Longer timeout when supported (newer Colab runtimes).
            try:
                drive.mount("/content/drive", force_remount=False, timeout_ms=300_000)
            except TypeError:
                drive.mount("/content/drive", force_remount=False)
            return True
        except (ValueError, OSError, RuntimeError) as e:
            last_err = e
            print(f"[drive] mount attempt {attempt}/3 failed: {e!r}")
            if attempt < 3:
                time.sleep(20)
    print("[drive] Giving up on Drive mount — next cell will use /content only.")
    print(f"[drive] Last error: {last_err!r}")
    return False

_MOUNT_OK = _mount_drive()

In [ ]:
import os

if os.path.isdir('/content/drive/MyDrive'):
    WORK_DIR = '/content/drive/MyDrive/colab/recsys_playground'
else:
    WORK_DIR = '/content/recsys_playground'

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

repo_url = 'https://github.com/allyoushawn/recsys_playground.git'
repo_dir = 'recsys_playground'

if not os.path.exists(repo_dir):
    !git clone {repo_url}

%cd {repo_dir}
!git pull origin main
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn requests papermill